<a href="https://colab.research.google.com/github/vanderbarbosa/mestrado/blob/main/04_Previsao_Machine_Learning_PETR4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# ==============================================================================
# PASSO 1: INTEGRAÇÃO E PREPARAÇÃO DO AMBIENTE
# ==============================================================================
!pip install arch xgboost scikit-learn pandas numpy --quiet

from google.colab import drive
import os
import pandas as pd
import numpy as np
from arch import arch_model
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Conectando ao Drive
drive.mount('/content/drive')
caminho_base = '/content/drive/MyDrive/Mestrado_PETR4/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# ==============================================================================
# PASSO 2: LEITURA DOS DADOS FINANCEIROS E CRIAÇÃO DO SENTIMENTO HISTÓRICO
# ==============================================================================
print("Carregando base financeira do Drive...")
df_master = pd.read_csv(caminho_base + "base_financeira_petr4.csv")

# Garantindo o formato de data (UTC)
df_master['Date'] = pd.to_datetime(df_master['Date'], utc=True).dt.date
df_master['Log_Retorno'] = df_master['Log_Retorno'] * 100

# MOCK DE DADOS (INJEÇÃO SINTÉTICA PARA VALIDAÇÃO DA ARQUITETURA DE ML)
# Como o web scraping coletou apenas os dias de hoje, vamos gerar um sentimento
# histórico simulado entre -1 e 1 para que os algoritmos tenham volume de dados para treinar.
print("AVISO: Injetando Índice de Sentimento Sintético (Mock) para os anos de 2018-2023...")
np.random.seed(42) # Garantindo reprodutibilidade acadêmica
df_master['Indice_Sentimento_Transformer'] = np.random.uniform(-1, 1, len(df_master))

print(f"Base preparada com {len(df_master)} dias de pregão para treinamento.")


Carregando base financeira do Drive...
AVISO: Injetando Índice de Sentimento Sintético (Mock) para os anos de 2018-2023...
Base preparada com 1486 dias de pregão para treinamento.


In [14]:
# ==============================================================================
# PASSO 3: MODELAGEM ECONOMÉTRICA DO RISCO (GARCH 1,1)
# ==============================================================================
print("\nModelando a volatilidade histórica da PETR4 com GARCH(1,1)...")
# O array agora não está vazio, então o GARCH conseguirá extrair a variância
modelo_garch = arch_model(df_master['Log_Retorno'], vol='Garch', p=1, q=1, dist='t')
resultado_garch = modelo_garch.fit(disp='off')
df_master['Volatilidade_GARCH'] = resultado_garch.conditional_volatility



Modelando a volatilidade histórica da PETR4 com GARCH(1,1)...


In [15]:
# ==============================================================================
# PASSO 4: ENGENHARIA DE ATRIBUTOS E DESLOCAMENTO TEMPORAL (Lags)
# ==============================================================================
# O modelo precisa prever D0 usando informações de D-1 (Ontem)
df_master['Retorno_Ontem'] = df_master['Log_Retorno'].shift(1)
df_master['Volatilidade_Ontem'] = df_master['Volatilidade_GARCH'].shift(1)
df_master['Sentimento_Ontem'] = df_master['Indice_Sentimento_Transformer'].shift(1)

# Alvo: 1 se a ação subiu, 0 se caiu
df_master['Alvo_Direcao'] = np.where(df_master['Log_Retorno'] > 0, 1, 0)
df_master.dropna(inplace=True)

In [16]:
# ==============================================================================
# PASSO 5: PREPARAÇÃO DO MACHINE LEARNING E TESTE CEGO
# ==============================================================================
X = df_master[['Retorno_Ontem', 'Volatilidade_Ontem', 'Sentimento_Ontem']]
Y = df_master['Alvo_Direcao']

# Separando 80% dos dados para treino e 20% para teste cronológico (walk-forward)
X_treino, X_teste, Y_treino, Y_teste = train_test_split(X, Y, test_size=0.20, shuffle=False)


In [17]:
# ==============================================================================
# PASSO 6: TREINAMENTO (SVM e XGBoost)
# ==============================================================================
print("\nTreinando SVM e XGBoost...")
modelo_svm = SVC(kernel='rbf', C=1.0, gamma='scale')
modelo_svm.fit(X_treino, Y_treino)
previsoes_svm = modelo_svm.predict(X_teste)

modelo_xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')
modelo_xgb.fit(X_treino, Y_treino)
previsoes_xgb = modelo_xgb.predict(X_teste)



Treinando SVM e XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:02:48] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [18]:
# ==============================================================================
# PASSO 7: RESULTADOS
# ==============================================================================
acuracia_svm = accuracy_score(Y_teste, previsoes_svm)
acuracia_xgb = accuracy_score(Y_teste, previsoes_xgb)

print("\n" + "="*50)
print("ACURÁCIA (TESTE DE ARQUITETURA - DADOS SINTÉTICOS)")
print("="*50)
print(f"SVM:     {acuracia_svm * 100:.2f}%")
print(f"XGBoost: {acuracia_xgb * 100:.2f}%")


ACURÁCIA (TESTE DE ARQUITETURA - DADOS SINTÉTICOS)
SVM:     55.56%
XGBoost: 53.87%
